Se calculó cuánto dinero se destinó al pago de comisiones por reservas canceladas del canal TA/TO. Este monto representa un gasto que el hotel podría evitar si modifica las condiciones de sus contratos con las agencias.

In [0]:
%sql

SELECT
    COUNT(*) AS reservas_canceladas,

    COUNT(*) * 1.5 AS perdida_marketing_usd

FROM cursodataacademy.hoteles.hotel_bookings

WHERE distribution_channel = 'TA/TO'
AND is_canceled = 1;

Se identificaron las reservas que fueron canceladas con menos de tres días de anticipación y se calculó el costo que representan para el hotel, considerando los USD 120 por cada cancelación.

In [0]:
%sql

WITH reservas AS (
    SELECT
        is_canceled,

        TO_DATE(
            CONCAT(
                arrival_date_day_of_month,
                '-',
                arrival_date_month,
                '-',
                arrival_date_year
            ),
            'd-MMMM-yyyy'
        ) AS arrival_date,

        TO_DATE(
            reservation_status_date,
            'd/M/yy'
        ) AS cancellation_date

    FROM cursodataacademy.hoteles.hotel_bookings
)

SELECT
    COUNT(*) AS cancelaciones_tardias,

    COUNT(*) * 120 AS perdida_operativa_usd

FROM reservas

WHERE is_canceled = 1
AND DATEDIFF(arrival_date, cancellation_date) BETWEEN 0 AND 2;

Se relacionaron las reservas con los países para conocer su procedencia, manteniendo todos los registros de la tabla principal aunque algunos no tengan un país relacionado.

In [0]:
%sql

CREATE OR REPLACE TABLE cursodataacademy.hoteles.tb_paises
USING DELTA
AS

SELECT DISTINCT
    country AS cod_pais,

    CASE country
        WHEN 'PRT' THEN 'Portugal'
        WHEN 'GBR' THEN 'Reino Unido'
        WHEN 'ESP' THEN 'España'
        WHEN 'FRA' THEN 'Francia'
        WHEN 'DEU' THEN 'Alemania'
        WHEN 'ITA' THEN 'Italia'
        WHEN 'IRL' THEN 'Irlanda'
        WHEN 'USA' THEN 'Estados Unidos'
        WHEN 'BRA' THEN 'Brasil'
        WHEN 'ARG' THEN 'Argentina'
        WHEN 'CHN' THEN 'China'
        WHEN 'JPN' THEN 'Japón'
        WHEN 'NLD' THEN 'Países Bajos'
        WHEN 'CHE' THEN 'Suiza'
        WHEN 'BEL' THEN 'Bélgica'
        WHEN 'CAN' THEN 'Canadá'
        WHEN 'AUS' THEN 'Australia'
        WHEN 'MEX' THEN 'México'
        WHEN 'PER' THEN 'Perú'
        ELSE country
    END AS nombre_pais

FROM cursodataacademy.hoteles.hotel_bookings

WHERE country IS NOT NULL;

In [0]:
%sql

SELECT
    r.is_canceled,
    p.nombre_pais

FROM cursodataacademy.hoteles.hotel_bookings r

LEFT JOIN cursodataacademy.hoteles.tb_paises p
    ON r.country = p.cod_pais;

Se obtuvieron los cinco países con mayor cantidad de reservas canceladas, lo que permite reconocer de qué mercados provienen la mayoría de estas cancelaciones.

In [0]:
%sql

SELECT
    p.nombre_pais,
    COUNT(*) AS total_cancelaciones

FROM cursodataacademy.hoteles.hotel_bookings AS r

INNER JOIN cursodataacademy.hoteles.tb_paises AS p
    ON r.country = p.cod_pais

WHERE r.is_canceled = 1

GROUP BY p.nombre_pais

ORDER BY total_cancelaciones DESC

LIMIT 5;

Se realizó una relación entre las reservas y los vendedores para revisar cómo están distribuidas las asignaciones e identificar si existen vendedores o reservas que no tienen una relación entre sí. Para ellos se crean tablas.

In [0]:
%sql

CREATE OR REPLACE TABLE cursodataacademy.hoteles.tb_vendedores
USING DELTA
AS

SELECT *
FROM VALUES
    (1, 'Carlos'),
    (2, 'María'),
    (3, 'José'),
    (4, 'Andrea'),
    (5, 'Luis')
AS vendedores(id_vendedor, nombre_vendedor);

In [0]:
%sql

CREATE OR REPLACE TABLE cursodataacademy.hoteles.tb_reservas_vendedores
USING DELTA
AS

SELECT
    distribution_channel,

    CASE
        WHEN ROW_NUMBER() OVER (ORDER BY arrival_date_year, arrival_date_month, arrival_date_day_of_month) % 4 = 1 THEN 1
        WHEN ROW_NUMBER() OVER (ORDER BY arrival_date_year, arrival_date_month, arrival_date_day_of_month) % 4 = 2 THEN 2
        WHEN ROW_NUMBER() OVER (ORDER BY arrival_date_year, arrival_date_month, arrival_date_day_of_month) % 4 = 3 THEN 3
        ELSE NULL
    END AS id_vendedor

FROM cursodataacademy.hoteles.hotel_bookings

LIMIT 100;

In [0]:
%sql

SELECT
    r.distribution_channel,
    v.nombre_vendedor

FROM cursodataacademy.hoteles.tb_reservas_vendedores AS r

FULL OUTER JOIN cursodataacademy.hoteles.tb_vendedores AS v

ON r.id_vendedor = v.id_vendedor;

Se relacionaron las reservas con las facturas, manteniendo todos los registros de facturación para poder identificar cuáles tienen información de hotel y cuáles no. Para ello se crean tablas

In [0]:
%sql

CREATE OR REPLACE TABLE cursodataacademy.hoteles.tb_facturas
USING DELTA
AS

SELECT *
FROM VALUES
    (1, 250.00),
    (2, 350.00),
    (3, 180.00),
    (4, 420.00),
    (5, 300.00),
    (999999, 150.00)
AS facturas(id_reserva, monto_facturado);

In [0]:
%sql

CREATE OR REPLACE TABLE cursodataacademy.hoteles.tb_reservas_facturas
USING DELTA
AS

SELECT
    ROW_NUMBER() OVER (
        ORDER BY arrival_date_year,
                 arrival_date_month,
                 arrival_date_day_of_month
    ) AS id_reserva,

    hotel

FROM cursodataacademy.hoteles.hotel_bookings

LIMIT 100;

In [0]:
%sql

SELECT
    f.id_reserva,
    f.monto_facturado,
    r.hotel

FROM cursodataacademy.hoteles.tb_reservas_facturas AS r

RIGHT JOIN cursodataacademy.hoteles.tb_facturas AS f

ON r.id_reserva = f.id_reserva;